# 02 — First End-to-End Run

Wire up the data pipeline, two agents, and the supervisor to produce a trading decision.

In [ ]:
import warnings
from pathlib import Path

from hmats.agents import RSIAgent, SMACrossoverAgent
from hmats.coordinator import Supervisor
from hmats.data.binance_store import fetch_and_store, load
from hmats.data.pipeline import build_snapshot, compute_indicators

warnings.filterwarnings("ignore", category=FutureWarning)

REPO_ROOT = Path.cwd().parents[2]
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd()
STORE_DIR = str(REPO_ROOT / "data" / "raw")

## 1. Load data

Load from the Binance Parquet store (populated by notebook 01 or fetched on the fly).

In [ ]:
TICKER = "BTCUSDT"

# Ensure store is populated
fetch_and_store(symbol=TICKER, interval="1h", start="2020-01-01", store_dir=STORE_DIR)

raw = load(symbol=TICKER, interval="1h", store_dir=STORE_DIR)
raw = raw.rename(columns={c: c.capitalize() for c in raw.columns})
df = compute_indicators(raw)

print(f"Data shape: {df.shape}")
df.tail(3)

## 2. Build a MarketSnapshot

In [ ]:
snapshot = build_snapshot(TICKER, df)

print(f"Ticker:    {snapshot.ticker}")
print(f"Timestamp: {snapshot.timestamp}")
print(f"OHLCV rows: {len(snapshot.ohlcv)}")
print(f"Indicators: {list(snapshot.indicators.keys())}")

## 3. Run the Supervisor

In [ ]:
supervisor = Supervisor(agents=[SMACrossoverAgent(), RSIAgent()])
decision = supervisor.run(snapshot)

print("\n" + "=" * 60)
print(f"DECISION: {decision.action.upper()}")
print(f"Confidence: {decision.confidence:.2f}")
print(f"Position size: {decision.position_size:.2f}")
print("=" * 60)

print("\nAgent signals:")
for agent_id, signal in decision.reasoning.items():
    print(f"  {agent_id}: {signal.action} (confidence={signal.confidence:.2f})")